# 04 - Robustness checks
The main result (notebook 02) tests only on 9 days of late September. Here we check whether the conclusions hold under other kinds of splits. These are checks only: no tuning, the same tuned Gradient Boosting settings every time.

1. **Blocked split inside every window:** all four seasons appear in training.
2. **Leave one window out:** train on three seasons, test on the fourth (a season the model has never seen).
3. **Unseen routes:** train on 12 routes, test on 4 routes the model has never seen.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from utils import HISTORY, TARGET, final_model, load_data, make_pipeline, planning_features

sns.set_theme(style="whitegrid")
df = load_data()
planning = planning_features(df)
weather = ["temperature_2m", "apparent_temperature", "relative_humidity_2m", "dew_point_2m", "precipitation",
           "snowfall", "pressure_msl", "cloud_cover", "wind_speed_10m", "wind_gusts_10m"]
feature_sets = {
    "planning": planning,
    "planning minus weather": [c for c in planning if c not in weather],
    "history": planning + HISTORY,
}

## Helper functions
- `baseline` = the training average for the same route, hour and weekday/weekend (same baseline as notebook 02). For unseen routes we drop the route from the key.
- `compare` fits each feature set on the training part and returns the MAE on the test part.

In [2]:
def baseline(train_part, test_part, keys=("bus_route", "hour", "is_weekend")):
    keys = list(keys)
    typical = train_part.groupby(keys)[TARGET].mean().rename("typical").reset_index()
    merged = test_part[keys].merge(typical, on=keys, how="left")
    pred = merged["typical"].fillna(train_part[TARGET].mean()).to_numpy()
    return mean_absolute_error(test_part[TARGET], pred)

def compare(train_part, test_part, baseline_keys=("bus_route", "hour", "is_weekend")):
    result = {"baseline": baseline(train_part, test_part, baseline_keys)}
    for name, features in feature_sets.items():
        pipe = make_pipeline(final_model(), features).fit(train_part[features], train_part[TARGET])
        result[name] = mean_absolute_error(test_part[TARGET], pipe.predict(test_part[features]))
    return result

## 1. Blocked split inside every window
In each 2-week window: first 10 days = train, last 4 days (Thursday to Sunday) = test. We use 4 days and not 2 because every window starts on a Monday, so the last 2 days would always be a weekend. The history features only look at the past, so no test values leak into the training features.

In [3]:
day_in_window = df.groupby("window_id")["transit_timestamp"].transform(lambda s: (s.dt.normalize() - s.dt.normalize().min()).dt.days)
blocked_train = df[day_in_window < 10]
blocked_test = df[day_in_window >= 10]
print(len(blocked_train), len(blocked_test))
blocked = pd.Series(compare(blocked_train, blocked_test), name="test MAE").round(1)
blocked

46080 18432


baseline                  59.6
planning                  49.2
planning minus weather    54.2
history                   35.3
Name: test MAE, dtype: float64

**Reading the blocked split:** when every season is in the training data, the planning model scores MAE 49.2 against 59.6 for the baseline (about 17% better), in line with the main result (16%). Removing weather makes it worse (54.2), and the history model is best (35.3).

## 2. Leave one window out
Each window (Jan, Apr, Jul, Sep) is held out in turn. This is the hardest test of whether the model carries over to a season it has not seen.

In [4]:
rows = []
for w in sorted(df["window_id"].unique()):
    result = compare(df[df["window_id"] != w], df[df["window_id"] == w])
    rows.append({"held-out window": w, **result})
loo = pd.DataFrame(rows).set_index("held-out window")
loo.loc["mean"] = loo.mean()
loo.round(1)

,baseline,planning,planning minus weather,history
held-out window,,,,
0,71.5,48.8,57.4,31.9
1,69.5,44.8,57.5,35.1
2,59.0,45.0,50.6,37.2
3,68.7,48.3,45.8,34.3
4,64.8,67.4,56.2,37.6
5,63.4,52.1,55.8,39.1
6,80.3,54.0,56.9,38.1
7,73.9,63.7,69.6,39.7
mean,68.9,53.0,56.2,36.6


**Reading leave-one-window-out:** each of the 8 two-week windows is predicted by a model that never saw it. The planning model averages MAE 53.0 against 68.9 for the baseline, which is **about 23% better**, and it beats the baseline in 7 of 8 windows. The exception is the June window (67.4 vs 64.8), a period before the ridership levels of some Queens routes changed, so the other windows disagree with it. The history model stays much better (36.6) because recent ridership tells it the current level.

**Weather:** removing weather makes the planning model worse in 6 of 8 held-out windows (average 56.2 vs 53.0), and in the blocked split too (54.2 vs 49.2). So weather helps even when it cannot be acting as a season label for that window. Still, do not read this as 'rain causes fewer riders': the weather columns follow a daily cycle, so they partly re-encode the time of day. We can say weather is a **useful input**, not that we proved a cause.

## 3. Unseen routes
`GroupKFold` keeps all rows of a route in the same fold, so the 4 test routes in each fold are completely new. Time is not split here, so this only tells us about new routes, not new dates. The baseline uses the hour and weekday/weekend average over all training routes.

In [5]:
rows = []
for fold, (tr, te) in enumerate(GroupKFold(n_splits=4).split(df, groups=df["bus_route"])):
    train_part, test_part = df.iloc[tr], df.iloc[te]
    result = compare(train_part, test_part, baseline_keys=("hour", "is_weekend"))
    rows.append({"fold": fold, "test routes": ", ".join(sorted(test_part["bus_route"].unique())),
                 "mean riders": test_part[TARGET].mean(), **result})
unseen = pd.DataFrame(rows).set_index("fold")
unseen.drop(columns="history").round(1)

,test routes,mean riders,baseline,planning,planning minus weather
fold,,,,,
0,"B41, BX19, M101, M86+, Q27, S79+",436.4,118.1,93.9,96.5
1,"B38, BX12+, BX9, M4, Q25, Q65",459.8,114.1,96.6,102.2
2,"B35, B8, BX7, M15+, Q10, Q58",509.1,178.4,150.7,152.0
3,"B1, B6, BX36, M14D+, M96, Q44+",444.6,116.3,104.5,105.1


**Reading unseen routes:** on routes the model never saw, the planning model averages MAE 111 against 132 for the baseline (about **15% better**), and it beats the baseline in all 4 folds. But the error is about 24% of average ridership, much worse than on known routes. The model learns each route's size from the route name and its GTFS facts, and it cannot guess the size of a new route well from 18 others.

**What this means for the app:** it should only offer the 24 routes it was trained on. Predicting a new route would need more routes in training (or route features such as nearby population).